In [ ]:
# Example 1: Instantiate the DjangoInstrumentor and verify its public API.
from opentelemetry.instrumentation.django import DjangoInstrumentor

instrumentor = DjangoInstrumentor()
print(f"type: {type(instrumentor).__name__}")
print(f"module: {type(instrumentor).__module__}")

# It should expose the BaseInstrumentor surface: instrument / uninstrument / is_instrumented_by_opentelemetry
for attr in ('instrument', 'uninstrument', 'instrumentation_dependencies'):
    assert hasattr(instrumentor, attr), f"DjangoInstrumentor missing {attr}"
    print(f"  has {attr}: {callable(getattr(instrumentor, attr))}")

assert type(instrumentor).__name__ == 'DjangoInstrumentor'


In [ ]:
# Example 2: instrumentation_dependencies declares which django versions are supported.
from opentelemetry.instrumentation.django import DjangoInstrumentor

instrumentor = DjangoInstrumentor()
deps = instrumentor.instrumentation_dependencies()
print(f"declared dependencies: {list(deps)}")

# Must be a non-empty iterable of PEP 440 specifier strings naming django.
assert deps, "instrumentation_dependencies returned an empty collection"
assert any('django' in d.lower() for d in deps), f"expected a django spec in {deps}"


In [ ]:
# Example 3: Instrument / uninstrument roundtrip via BaseInstrumentor.
from opentelemetry.instrumentation.django import DjangoInstrumentor

instrumentor = DjangoInstrumentor()

# Begin from a clean state in case a prior cell left it instrumented.
if instrumentor.is_instrumented_by_opentelemetry:
    instrumentor.uninstrument()

instrumentor.instrument()
print("instrumented OK")

instrumentor.uninstrument()
print("uninstrumented OK")

# Repeated uninstrument should not raise.
instrumentor.uninstrument()
assert True

In [ ]:
# Example 4: Inspect the Django request/response middleware classes shipped with this package.
from opentelemetry.instrumentation.django.middleware import otel_middleware

# The submodule exposes a middleware class that Django can install.
candidates = [
    getattr(otel_middleware, name) for name in dir(otel_middleware)
    if name.endswith('Middleware') and isinstance(getattr(otel_middleware, name), type)
]
print(f"middleware classes found: {[c.__name__ for c in candidates]}")

assert candidates, "no middleware class exported from opentelemetry.instrumentation.django.middleware.otel_middleware"

# Each middleware class follows the Django (get_response) constructor protocol.
for cls in candidates:
    assert hasattr(cls, '__init__')
    print(f"  {cls.__name__}: {cls.__module__}")


In [ ]:
# Example 5: Version metadata + entry-point shape.
import opentelemetry.instrumentation.django as oti_django
from opentelemetry.instrumentation.django.package import _instruments
from opentelemetry.instrumentation.django.version import __version__

print(f"opentelemetry-instrumentation-django version: {__version__}")
print(f"_instruments specifier: {_instruments}")

# These constants drive autoinstrumentation discovery; both must be present and non-empty.
assert isinstance(__version__, str) and __version__
assert _instruments, "expected _instruments to enumerate supported django versions"
assert any('django' in entry.lower() for entry in _instruments)

# Top-level package re-exports DjangoInstrumentor as the public entry point.
assert hasattr(oti_django, 'DjangoInstrumentor')
print("DjangoInstrumentor is the public entry point: OK")
